# Five-PDF reasoning pilot audit
Reads the local scouting database for batch `20260925T211558Z-4b87b736`. Each row is one PDF and one requested reasoning level. Token totals come from provider responses; warning counts are quote/shape checks, not measured transcription error rates. Run from the repository root in the uv environment. No API calls are made.

In [ ]:
import json
from pathlib import Path
import duckdb
from scout import reasoning_tokens, usage_counts

batch = '20260925T211558Z-4b87b736'
with duckdb.connect(str(Path('data/scout.duckdb')), read_only=True) as db:
    rows = db.execute('''
        SELECT d.ordinal, d.title, a.pdf_sha256, a.request_json, a.parsed_json,
               a.usage_json, a.validation_warnings
        FROM attempts a JOIN batch_documents d
          ON a.batch_id = d.batch_id AND a.pdf_sha256 = d.pdf_sha256
        WHERE a.batch_id = ? AND a.status = 'ok'
        ORDER BY d.ordinal
    ''', [batch]).fetchall()

for ordinal, title, sha, request, parsed, usage_raw, warnings_raw in rows:
    level = json.loads(request).get('reasoning_effort', 'default')
    claim = json.loads(parsed)
    usage = json.loads(usage_raw) if usage_raw else None
    counts = usage_counts(usage)
    processes = claim.get('processes', [])
    dockets = sorted({d for p in processes for d in p.get('dockets', [])})
    project = claim.get('project') or {}
    print(ordinal, level, title, 'dockets=', dockets,
          'units=', project.get('proposed_units'),
          'entities=', len(claim.get('entities', [])),
          'events=', sum(len(p.get('events', [])) for p in processes),
          'questions=', sum(len(p.get('questions', [])) for p in processes),
          'warnings=', len(json.loads(warnings_raw or '[]')),
          'input=', counts[0] if counts else None,
          'cached_input=', counts[1] if counts else None,
          'output=', counts[2] if counts else None,
          'reasoning=', reasoning_tokens(usage),
          'total=', counts[3] if counts else None)


In [ ]:
# High-effort chronology with provenance classification for join review.
for ordinal, title, sha, request, parsed, *_ in rows:
    if json.loads(request).get('reasoning_effort') != 'high':
        continue
    print('\n', ordinal, title, sha[:12])
    for process in json.loads(parsed).get('processes', []):
        print(' ', process.get('kind'), process.get('dockets', []))
        for event in process.get('events', []):
            print('   ', event.get('date'), event.get('kind'),
                  event.get('effect'), event.get('reported_as'),
                  event.get('summary'))
